In [4]:
import pandas as pd
from dotenv import dotenv_values
import requests
from io import BytesIO

In [5]:
config = dotenv_values(dotenv_path=".env")

# Pipeline

In [7]:
url = config["DATA_TEST_URL"]
headers = {"Authorization": f"Bearer {config['GITHUB_TOKEN']}"}
r = requests.get(url, headers=headers, timeout=30)
r.raise_for_status()  # Check if the request was successful

df = pd.read_excel(BytesIO(r.content), header=None, sheet_name=None, engine="openpyxl")

In [8]:
df

In [17]:
cleaned_sheets = {}
for name, sheet in df.items():
    sheet = sheet.dropna(how="all")       # eliminar filas vacías
    sheet.columns = sheet.iloc[0]         # primera fila como headers
    sheet = sheet.drop(sheet.index[0])    # eliminar fila usada como header
    sheet = sheet.reset_index(drop=True)
    cleaned_sheets[name] = sheet

In [18]:
cleaned_sheets.keys()

In [19]:
#MAPA DE VALORES COGNITIVOS

dominios_cognitivos = {
        "prueba_validez": [
            'reconocimiento_inmediato_ensayo_1', 'reconocimiento_inmediato_ensayo_2', 'reconocimiento_diferido_ensayo_3'
            ],

        "orientacion": [
            'orientacion_persona',
            'orientacion_espacio',
            'orientacion_tiempo'
            ],

        "cap_atencional": [
            'vel_procesamiento_atencion_sostenida_auditiva','vel_procesamiento_atencion_sostenida_visual','vel_procesamiento_atencion_selectiva_visual','vel_procesamiento_atencion_dividida_visual'
            ],

        "lenguaje": [
            'denominacion_imagenes',
            'comprension_ordenes',
            'material_verbal_complejo',
            'lectura_palabras'
            ],

        "cvlt": [
            'evocacion_inmediata_lista_a',
            'recuerdo_inmediato_lista_a',
            'recuerdo_inmediato_lista_b',
            'recuerdo_libre_corto_plazo',
            'recuerdo_libre_largo_plazo',
            'recuerdo_libre_largo_plazo_clave',
            'reconocimiento'
            ],

        "memoria_visual": [
            'visual_evocacion_diferida',
            'visual_reconocimiento_inmediato',
            'visual_reconocimiento_diferido'
            ],

        "capacidad_visuoperceptiva": ['imagenes_superpuesta'],

        "praxis": [
            'constructiva',
            'gestos_ideomotora_simbolicos_orden_derecha',
            'gestos_ideomotora_simbolicos_orden_izquierda',
            'gestos_ideomotora_simbolicos_imitacion_derecha','gestos_ideomotora_simbolicos_imitacion_izquierda'
            ],

        "funcionalidades_ejecutivas": [
            'memoria_de_trabajo_digitos_inversos',
            'memoria_de_trabajo_digitos_secuenciales','test_fluidez_de_fonologica_FAS_F',
            'test_fluidez_de_fonologica_FAS_A',
            'test_fluidez_de_fonologica_FAS_S',
            'evocacion_categorial_semantica',
            'semejanzas',
            'matrices',
            'comprension_abstraccion',
            'atencion_alternante',
            'stroop_A',
            'stroop_b',
            'stroop_c'
            ]
}

In [20]:
test = [
    'orientacion_en_persona',
    'orientacion_en_espacio',
    'orientacion_en_tiempo',

    'atencion_sostenida_auditiva',
    'atencion_sostenida_visual',
    'atencion_selectiva_visual',
    'atencion_dividida_visual',

    'denominacion_de_imagenes',
    'comprension_de_ordenes',
    'material_verbal_complejo',

    'evocacion_inmediata_lista_a',
    'recuerdo_inmediato_lista_a',
    'recuerdo_inmediato_lista_b',
    'recuerdo_libre_a_corto_plazo',
    'recuerdo_libre_a_largo_plazo',
    'recuerdo_libre_a_largo_plazo_clave',
    'reconocimiento',

    'evocacion_diferida',
    'reconocimiento_inmediato_ensayo_2',
    'reconocimiento_diferido_ensayo_3',

    'imagenes_superpuesta',

    'constructiva',
    'ideomotora_gestos_simbolicos_a_la_orden_derecha',
    'ideomotora_gestos_simbolicos_a_la_orden_izquierda',
    'ideomotora_gestos_simbolicos_a_la_imitacion_derecha',
    'ideomotora_gestos_simbolicos_a_la_imitacion_izquierda',

    'memoria_de_trabajo_digitos_inversos',
    'memoria_de_trabajo_digitos_secuenciales',
    'test_de_fluidez_fonologica_fas_f',
    'test_de_fluidez_fonologica_fas_a',
    'test_de_fluidez_fonologica_fas_s',
    'evocacion_categorial_semantica',
    'semejanzas',
    'matrices',
    'comprension_abstraccion',
    'atencion_alternante',
    'stroop_a',
    'stroop_b',
    'stroop_c',
    'stroop_palabra',
    'stroop_colores',
    'stroop_interferencia',
  ]

In [21]:
cleaned_sheets['015-DCM-H62']

In [22]:
def clean_value(cadena):
  cadena = cadena.strip().lower().replace(' ', '_').replace('-','').replace("á", "a").replace("é", "e").replace("í", "i").replace("ó", "o").replace("ú", "u").replace(".","").replace("(","").replace(")","")
  return cadena

In [23]:
#FUNCION PARA LIMPIAR KEYS
def clean_keys(df):
  headers = {}
  for header in df.keys():
    new_header = clean_value(header)
    headers[header]=new_header
  df = df.rename(columns = headers)
  return df

In [48]:
def search_values(df_raw, df_clean):
  df_raw = clean_keys(df_raw)
  patient_data = {}
  for column in df_clean:
    # print(f'buscando el valor de la columna {column}')
    column_keys = df_raw[df_raw.columns[0]]
    column_keys_clean = column_keys.apply(clean_value)
    filter = column_keys_clean.str.contains(column, case=False,na=False)
    if(len(df_raw[filter].index) > 0):
      idx = df_raw[filter].index[0]
      finding_values = df_raw.iloc[idx, df_raw.columns.get_loc("percentil_o_p_directa")]
      # print(f'valor: {finding_values}')
      patient_data[column] = finding_values
    else:
      patient_data[column] = None
  return patient_data

In [25]:
pd.set_option("display.max_columns", None)  # mostrar todas las columnas

In [26]:
patients_info = []
for i, patient in enumerate(cleaned_sheets):
  print(patient)
  try:
    patient_data = search_values(cleaned_sheets[patient], test, i)
    patient_data['sexo'] = 'M' if 'H' in patient else 'F'
    patient_data['DC'] = 0 if 'NORMAL' in patient else 1 if 'DCL' in patient else 2
    patients_info.append(patient_data)
  except Exception as e:
    print(f'ocurrio un error {e}')

In [27]:
patients_df = pd.DataFrame(patients_info)
patients_df

## DC

In [28]:
url = config["DATA_TEST_URL"]

headers = {"Authorization": f"Bearer {config['GITHUB_TOKEN']}"}
r = requests.get(url, headers=headers, timeout=30)
r.raise_for_status()  # Check if the request was successful

xlsx_data = pd.read_excel(BytesIO(r.content), header=None, sheet_name=None, engine="openpyxl")

In [50]:
cleaned_sheets_dc = {}
for name, sheet in xlsx_data.items():
    sheet = sheet.dropna(how="all")       # eliminar filas vacías
    sheet.columns = sheet.iloc[0]         # primera fila como headers
    sheet = sheet.drop(sheet.index[0])    # eliminar fila usada como header
    sheet = sheet.reset_index(drop=True)
    cleaned_sheets_dc[name] = sheet
    
dc_data = []
for i, patient in enumerate(cleaned_sheets_dc):
  print(patient)
  try:
    patient_data = search_values(cleaned_sheets_dc[patient], test)
    patient_data['dc'] = 0 if 'F06' in patient else 1 if 'F02' in patient else 2
    patient_data['sheet_name'] = patient
    dc_data.append(patient_data)
  except Exception as e:
    # data_map = dict(zip(df[0], df[1]))
    print(cleaned_sheets_dc[patient].columns)
    print(f'ocurrio un error {e}')
    

In [30]:
dc_df = pd.DataFrame(dc_data)
dc_df

In [44]:


cleaned_sheets_dc = {}
for name, sheet in xlsx_data.items():
    sheet = sheet.dropna(how="all")       # eliminar filas vacías
    sheet.columns = sheet.iloc[0]         # primera fila como headers
    sheet = sheet.drop(sheet.index[0])    # eliminar fila usada como header
    sheet = sheet.reset_index(drop=True)
    cleaned_sheets_dc[name] = sheet
    
dc_data = []
for i, patient in enumerate(cleaned_sheets_dc):
  print(patient)
  try:
    patient_data = search_values(cleaned_sheets_dc[patient], test, i)
    # patient_data['sexo'] = 'M' if 'H' in patient else 'F'
    patient_data['dc'] = 0 if 'F06' in patient else 1 if 'F02' in patient else 2
    patient_data['sheet_name'] = patient
    dc_data.append(patient_data)
  except Exception as e:
    data_map = dict(zip(df[0], df[1]))
    print(i)
    print(f'ocurrio un error {e}')

In [ ]:
data = dict(zip(df[0], df[1]))

In [37]:
s60 = pd.DataFrame(data, index=[0])